In [12]:
import os
import sys
from dotenv import load_dotenv
load_dotenv()
project_dir = str(os.environ.get("PROJECT_ROOT"))

utils_path = os.path.join(project_dir, "LangGraph", "src")
print(utils_path)
sys.path.append(os.path.abspath(utils_path))
viet_ocr_correction = os.path.join(project_dir, "VietnameseOcrCorrection")
sys.path.append(os.path.abspath(viet_ocr_correction))
print(viet_ocr_correction)

/Users/jajajou1778/UIT_DOCS_AGENT/LangGraph/src
/Users/jajajou1778/UIT_DOCS_AGENT/VietnameseOcrCorrection


In [13]:
from tool.predictor import Predictor

In [14]:
from agent.utils import predict

In [17]:
predict("N ghevàĐc")

'N ghe và Đc'

In [48]:
import re, unicodedata
from pathlib import Path

md_path = Path("/Users/jajajou1778/UIT_DOCS_AGENT/data/MinerU/547-qd-dhcntt_30-8-2019_qui_dinh_dao_tao_ngoai_ngu_doi_voi_he_chinh_qui_khoa_2019_0_0/c7037cb5-cbd8-49ff-93cc-f234dbb84004/547-qd-dhcntt_30-8-2019_qui_dinh_dao_tao_ngoai_ngu_doi_voi_he_chinh_qui_khoa_2019_0_0/auto/547-qd-dhcntt_30-8-2019_qui_dinh_dao_tao_ngoai_ngu_doi_voi_he_chinh_qui_khoa_2019_0_0.md")
text = md_path.read_text(encoding="utf-8")

# --- utilities ---------------------------------------------------------------

HAS_LETTERS = re.compile(r"[A-Za-zÀ-ỹĐđ]")

def predict_safe(s: str) -> str:
    # Only send real text (not just whitespace/punct) to your model
    if not s or not HAS_LETTERS.search(s):
        return s
    cand = s.strip()
    if not cand:
        return s
    try:
        out = predict(cand)
        # restore original leading/trailing whitespace from s
        lead_ws = re.match(r"^\s*", s).group(0)
        trail_ws = re.search(r"\s*$", s).group(0)
        return lead_ws + out + trail_ws
    except Exception:
        return s

# --- HTML table handler ------------------------------------------------------

TABLE_BLOCK_RE = re.compile(r"<table[\s\S]*?</table>", flags=re.IGNORECASE)
TAG_SPLIT_RE   = re.compile(r"(<[^>]+>)")

def fix_html_table_block(block: str) -> str:
    """Run predict only on text nodes inside an HTML <table>…</table> block."""
    parts = TAG_SPLIT_RE.split(block)
    fixed = []
    for part in parts:
        if not part:
            continue
        if part.startswith("<") and part.endswith(">"):
            fixed.append(part)  # keep tag/attrs as-is
        else:
            # This is a text node between tags; preserve exact spacing
            # Split further on runs of whitespace to keep layout identical
            subparts = re.split(r"(\s+)", part)
            for sp in subparts:
                if not sp or sp.isspace():
                    fixed.append(sp)
                else:
                    fixed.append(predict_safe(sp))
    return "".join(fixed)

# --- Markdown pipe table handler --------------------------------------------

PIPE_ROW_RE = re.compile(r"^\|.*\|.*$", flags=re.MULTILINE)

def fix_pipe_table_row(row: str) -> str:
    """
    Apply predict to cell contents while preserving delimiters/spaces.
    We split on '|' but keep them, and we don’t touch alignment rows (---, :---:, etc.).
    """
    # alignment row? leave it
    if re.match(r"^\|\s*[:\-| ]+\s*\|$", row.strip()):
        return row
    # split keeping the pipes
    parts = re.split(r"(\|)", row)
    out = []
    for p in parts:
        if p == "|":
            out.append(p)
            continue
        if p.strip() == "":
            out.append(p)
            continue
        # within a cell, keep surrounding spaces, fix inner text
        m = re.match(r"^(\s*)(.*?)(\s*)$", p, flags=re.DOTALL)
        lead, core, trail = m.groups()
        out.append(lead + predict_safe(core) + trail)
    return "".join(out)

# --- main chunker: protect code/math/headers/lists, process text -------------
PROTECT_RE = re.compile(
    r"("
    r"```[\s\S]*?```"                   # fenced code
    r"|`[^`\n]+`"                       # inline code
    r"|\$[^$\n]*\$"                     # inline math $...$
    r"|\\$begin:math:text$[^$end:math:text$]*\\\)"                  # $begin:math:text$ ... $end:math:text$
    r"|^[ \t]{0,3}#{1,6}[ \t].*$"       # headings
    r"|^[ \t]{0,3}[-*+][ \t].*$"        # bullets
    r"|^[ \t]{0,3}\d{1,3}[.)][ \t].*$"  # ordered
    r")",
    flags=re.MULTILINE
)

def iter_chunks(s: str):
    last = 0
    for m in TABLE_BLOCK_RE.finditer(s):  # first, isolate <table>…</table> blocks
        # Everything before this table still needs processing (non-table path)
        if m.start() > last:
            yield ("other", s[last:m.start()])
        yield ("html_table", m.group(0))
        last = m.end()
    if last < len(s):
        yield ("other", s[last:])

def process_non_table(s: str) -> str:
    # Within non-table blocks, protect code/math/headers/lists, but allow pipe tables.
    out = []
    last = 0
    for m in PROTECT_RE.finditer(s):
        if m.start() > last:
            out.append(process_text_and_pipe_tables(s[last:m.start()]))
        out.append(m.group(0))  # keep protected chunk as-is
        last = m.end()
    if last < len(s):
        out.append(process_text_and_pipe_tables(s[last:]))
    return "".join(out)

def process_text_and_pipe_tables(s: str) -> str:
    # Fix pipe table rows line-by-line, and normal text paragraphs with predict
    lines = s.splitlines(keepends=True)
    out = []
    for line in lines:
        if PIPE_ROW_RE.match(line):
            out.append(fix_pipe_table_row(line))
        else:
            # normal text: split by double newlines already preserved by splitlines
            out.append(predict_safe(line))
    return "".join(out)

def preserve_layout_fix(s: str) -> str:
    chunks = []
    for kind, seg in iter_chunks(s):
        if kind == "html_table":
            chunks.append(fix_html_table_block(seg))
        else:  # non-table area
            chunks.append(process_non_table(seg))
    return "".join(chunks)

# ---- run --------------------------------------------------------------------
fixed = preserve_layout_fix(text)
fixed = unicodedata.normalize("NFC", fixed)  # compose diacritics; layout untouched
md_path.write_text(fixed, encoding="utf-8", newline="\n")
print("✅ Done. Text fixed; tables (HTML & pipe) preserved; layout intact.")

✅ Done. Text fixed; tables (HTML & pipe) preserved; layout intact.
